# Real Data Analysis & Preprocessing

This notebook covers exploratory data analysis and the full preprocessing pipeline for the real sensor and session data extracted from our database.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.preprocessing import StandardScaler
import sys

%matplotlib inline
sns.set_theme(style="whitegrid")

# Setup Paths
MAL_DIR = Path.cwd().parent
if MAL_DIR.name != "MAL":
    MAL_DIR = next(path for path in [Path.cwd(), *Path.cwd().parents] if path.name == "MAL")
DATA_DIR = MAL_DIR / "data" / "real"

SENSOR_PATH = DATA_DIR / "sensor_history.csv"
SESSIONS_PATH = DATA_DIR / "sessions.csv"


## 1. Load Data

In [ ]:
sensor_df = pd.read_csv(SENSOR_PATH, parse_dates=['sent_at'])
sessions_df = pd.read_csv(SESSIONS_PATH, parse_dates=['started_at', 'last_pulse_at'])

print(f"Loaded {len(sensor_df)} sensor records.")
print(f"Loaded {len(sessions_df)} session records.")
display(sensor_df.head())
display(sessions_df.head())

## 2. Drop Sessions without Target (study_quality)
We drop any sessions that do not have a recorded `study_quality` label, as they cannot be used for supervised training. If we tried to imputed them this woul lead to data leakage.

In [ ]:
# Filter sessions
valid_sessions = sessions_df.dropna(subset=['study_quality']).copy()
print(f"Valid sessions with study_quality: {len(valid_sessions)} (dropped {len(sessions_df) - len(valid_sessions)})")

# Note: Currently, the real db export might have 0 valid sessions. 
# If so, the following steps will operate on empty DataFrames.
valid_session_ids = valid_sessions['id'].unique()
sensor_df = sensor_df[sensor_df['session_id'].isin(valid_session_ids)].copy()


## 3. Analysis
#### TODO after getting real usable data
#### TODO2 we need to decide what to do with missing noise column

In [ ]:
if not sensor_df.empty:
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    sns.histplot(sensor_df['temperature'], kde=True, ax=axes[0, 0]).set_title('Temperature Distribution')
    sns.histplot(sensor_df['humidity'], kde=True, ax=axes[0, 1]).set_title('Humidity Distribution')
    sns.histplot(sensor_df['co2_level'], kde=True, ax=axes[1, 0]).set_title('CO2 Distribution')
    sns.histplot(sensor_df['light_level'], kde=True, ax=axes[1, 1]).set_title('Light Distribution')
    plt.tight_layout()
    plt.show()
else:
    print("No sensor data available for valid sessions to plot.")

## 4. Preprocessing Pipeline

In [ ]:
## A. Missing Values Handling

# Sort chronologically per session
sensor_df = sensor_df.sort_values(by=['session_id', 'sent_at'])

# Forward fill missing sensor data, then backward fill for edge cases
cols_to_fill = ['temperature', 'humidity', 'co2_level', 'light_level']
sensor_df[cols_to_fill] = sensor_df.groupby('session_id')[cols_to_fill].ffill().bfill()


In [ ]:
## B. Outliers Handling (Clipping)

# Clip to realistic physical bounds
sensor_df['temperature'] = sensor_df['temperature'].clip(lower=10, upper=40)
sensor_df['humidity'] = sensor_df['humidity'].clip(lower=0, upper=100)
sensor_df['co2_level'] = sensor_df['co2_level'].clip(lower=400, upper=5000)
sensor_df['light_level'] = sensor_df['light_level'].clip(lower=0, upper=2000)


In [ ]:
## C. Scaling
scaler = StandardScaler()
cols_to_scale = ['temperature', 'humidity', 'co2_level', 'light_level']
if not sensor_df.empty:
    sensor_df[cols_to_scale] = scaler.fit_transform(sensor_df[cols_to_scale])


In [ ]:
## D. Simulate and One-Hot Encode student_type
# TODO: In the future, this column will be merged from a real external CSV.
# For now, we simulate assigning a student type to each valid session.
student_types_list = ['visual', 'auditory', 'kinesthetic', 'reading_writing']

if not valid_sessions.empty:
    np.random.seed(42)
    simulated_student_types = pd.DataFrame({
        'id': valid_sessions['id'],
        'student_type': np.random.choice(student_types_list, size=len(valid_sessions))
    })
    
    # Merge into valid_sessions
    valid_sessions = valid_sessions.merge(simulated_student_types, on='id', how='left')
    
    # One-hot encode
    valid_sessions = pd.get_dummies(valid_sessions, columns=['student_type'], prefix='student_type', drop_first=False)
    
    display(valid_sessions.head())
else:
    print("No valid sessions to simulate student_type on.")


## 5. Linearize and Save ML Dataset

In [ ]:
# Import our external linearization script
sys.path.append(str(MAL_DIR))
from scripts.linearize_and_glue import linearize_and_glue

# Pass the fully preprocessed dataframes directly to the script function
final_ml_dataset = linearize_and_glue(sensor_df, valid_sessions)

display(final_ml_dataset.head())

# Save the final dataset
output_path = DATA_DIR / "linearized_sessions_with_target.csv"
final_ml_dataset.to_csv(output_path, index=False)
print(f"Saved final dataset ({len(final_ml_dataset)} rows) to {output_path}")